# 4.2 — Cadastro mestre de clientes (KNA1)

- **Propósito:** Consolidar os dados gerais de clientes provenientes do SAP KNA1.
- **Entrada:** Arquivos XLS/XLSX do volume `dm_customers/kna1_sap`
- **Saída:** `parts_hdbk_sandbox.dm_customers.kna1_sap`
- **Chave:** Cliente · **Carga:** Completa

In [0]:
import pandas as pd
import glob
from pyspark.sql import functions as F

# Bibliotecas para leitura de arquivos Excel e manipulação de dados

In [0]:
# Instalar biblioteca para leitura de arquivos XLSX
%pip install openpyxl --quiet

In [0]:
# Caminho do volume com os arquivos XLSX
VOLUME_PATH = "/Volumes/parts_hdbk_sandbox/dm_customers/kna1_sap"

# Ler todos os arquivos XLSX do volume usando pandas e converter para Spark DF
xls_files = glob.glob(f"{VOLUME_PATH}/*.xls*")
print(f"Arquivos encontrados: {len(xls_files)}")
for f in xls_files:
    print(f"  - {f}")

# Concatenar todos os arquivos em um único pandas DataFrame
pdf_list = []
for f in xls_files:
    pdf_part = pd.read_excel(f, dtype=str)
    pdf_list.append(pdf_part)
    print(f"  {f}: {len(pdf_part)} linhas")

pdf = pd.concat(pdf_list, ignore_index=True)
print(f"\nTotal de linhas lidas: {len(pdf)}")

# Converter para Spark DataFrame
df_kna1 = spark.createDataFrame(pdf)

print(f"\nSchema:")
df_kna1.printSchema()
print(f"\nAmostra:")
display(df_kna1.limit(10))

In [0]:
import re
import unicodedata


def normalize_col_name(name: str) -> str:
    """Normaliza nome de coluna: remove acentos, lowercase, troca caracteres
    especiais por underscore e remove underscores extras."""
    # Remover acentos
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    # Lowercase
    name = name.lower()
    # Substituir caracteres não-alfanuméricos por underscore
    name = re.sub(r"[^a-z0-9]+", "_", name)
    # Remover underscores no início/fim
    name = name.strip("_")
    return name


# =============================================================================
# MAPA DE RENOMEAÇÃO DE COLUNAS
# -----------------------------------------------------------------------------
# Formato: "nome_normalizado": "novo_nome_desejado"
# Se a coluna normalizada existir neste mapa, será renomeada para o valor.
# Colunas sem entrada no mapa mantêm o nome normalizado.
# =============================================================================
COLUMN_RENAME_MAP = {
    # "nome_normalizado_original": "novo_nome",
    # Exemplos:
    "ps": "pais",
    "local": "cidade",
    "rg": "estado",
    "nome_3": "razao_social",
}


# Aplicar normalização a todas as colunas
original_cols = df_kna1.columns
normalized_cols = [normalize_col_name(c) for c in original_cols]

print("Mapeamento de colunas:")
for orig, norm in zip(original_cols, normalized_cols):
    # Aplicar rename do mapa se houver entrada; caso contrário, manter normalizado
    final_name = COLUMN_RENAME_MAP.get(norm, norm)
    df_kna1 = df_kna1.withColumnRenamed(orig, final_name)
    if norm != final_name:
        print(f"  {orig:15s} -> {norm:15s} -> {final_name}  (renomeado via mapa)")
    else:
        print(f"  {orig:15s} -> {final_name}")

# --- Transformações de negócio ---

# Remover zeros à esquerda da coluna cliente
col_cliente = COLUMN_RENAME_MAP.get("cliente", "cliente")
df_kna1 = df_kna1.withColumn(
    col_cliente,
    F.regexp_replace(F.col(col_cliente), r"^0+", "")
)

print("\nTransformações aplicadas:")
print("  - Colunas normalizadas (lowercase, sem caracteres especiais).")
print("  - Mapa de renomeação aplicado (colunas com entrada no COLUMN_RENAME_MAP renomeadas).")
print("  - Zeros à esquerda removidos da coluna 'cliente'.")
print("\nSchema final:")
df_kna1.printSchema()
print("Amostra:")
display(df_kna1.limit(10))

In [0]:
DEST_TABLE = "parts_hdbk_sandbox.dm_customers.kna1_sap"

# Dropar tabela antiga se existir
spark.sql(f"DROP TABLE IF EXISTS {DEST_TABLE}")

# Criar tabela Delta
df_kna1.write.saveAsTable(DEST_TABLE)

# Tabela criada com sucesso

In [0]:
%sql
-- Definir coluna NOT NULL (requisito para PK)
ALTER TABLE parts_hdbk_sandbox.dm_customers.kna1_sap
ALTER COLUMN cliente SET NOT NULL;

-- Adicionar constraint de chave primaria
ALTER TABLE parts_hdbk_sandbox.dm_customers.kna1_sap
ADD CONSTRAINT pk_kna1_sap PRIMARY KEY (cliente);

In [0]:
# Aplicar comentários e tags do Unity Catalog na tabela e colunas
DEST_TABLE = "parts_hdbk_sandbox.dm_customers.kna1_sap"

# Comentário da tabela
TABLE_COMMENT = """
Tabela de dados gerais do cliente (KNA1 SAP) com informacoes de pais, localidade, regiao e nome.
Ingestao a partir de arquivo XLSX do volume kna1_sap.

Chave Primaria: cliente
Atualizacao: Carga manual via volume.

Colunas (campo SAP original -> nome final):
  - cliente -> codigo do cliente SAP
  - ps -> pais: pais do cliente
  - local -> cidade: cidade/localidade do cliente
  - rg -> estado: regiao/estado do cliente
  - nome_3 -> razao_social: nome/razao social do cliente
"""

spark.sql(f"""
    COMMENT ON TABLE {DEST_TABLE} IS '{TABLE_COMMENT.replace(chr(39), chr(39)+chr(39))}'
""")

# Tags do Unity Catalog
spark.sql(f"""
    ALTER TABLE {DEST_TABLE} SET TAGS (
        'domain' = 'customers',
        'layer' = 'refined',
        'source' = 'sap',
        'source_table' = 'KNA1',
        'data_classification' = 'internal'
    )
""")

# Propriedades customizadas
spark.sql(f"""
    ALTER TABLE {DEST_TABLE} SET TBLPROPERTIES (
        'business_owner' = 'Demand Planning',
        'technical_owner' = 'Andre Causs',
        'data_domain' = 'Customer',
        'source_system' = 'SAP',
        'source_path' = '/Volumes/parts_hdbk_sandbox/dm_customers/kna1_sap',
        'refresh_frequency' = 'manual_volume_upload',
        'primary_key' = 'cliente'
    )
""")

# Comentários nas colunas (usar nomes finais após renomeação)
COLUMN_COMMENTS = {
    "cliente": "Codigo do cliente SAP (sem zeros a esquerda). Chave primaria. Campo original: CLIENTE.",
    "pais": "Pais do cliente (ex: BR=Brasil). Campo original SAP: PS.",
    "cidade": "Cidade/localidade do cliente. Campo original SAP: LOCAL.",
    "estado": "Regiao/estado do cliente (ex: SP, RJ, CE). Campo original SAP: RG.",
    "razao_social": "Nome ou razao social do cliente. Campo original SAP: NOME_3.",
}

for column_name, comment in COLUMN_COMMENTS.items():
    escaped_comment = comment.replace("'", "''")
    spark.sql(f"COMMENT ON COLUMN {DEST_TABLE}.{column_name} IS '{escaped_comment}'")

print(f"Metadados completos aplicados a tabela {DEST_TABLE}")